# 03 — CNN-LSTM Model Training

Trains the hybrid CNN-LSTM classifier on T&FD features extracted from DAS signals.

**Architecture:** `Conv1D → MaxPool → Dropout → LSTM → Dense → Softmax`  
**Best result (T&FD):** 97% train-position detection rate  
**Published in:** *Green Energy and Intelligent Transportation*, Elsevier 2024  
**DOI:** [10.1016/j.geits.2024.100178](https://doi.org/10.1016/j.geits.2024.100178)

## 1. Load & Prepare Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

DATA_PATH = "data/processed/Training_Data.csv"
data = pd.read_csv(DATA_PATH)
print(f"Loaded: {data.shape}")
print(data[data.columns[0]].value_counts())

In [ ]:
X = data.drop(data.columns[0], axis=1)
y = data[data.columns[0]]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Label mapping:")
for orig, enc in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"  {orig} -> {enc}")

# SMOTE — handle class imbalance (NC >> AC1, AC2)
print(f"Before SMOTE: {dict(zip(*np.unique(y_encoded, return_counts=True)))}")
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y_encoded)
print(f"After SMOTE:  {dict(zip(*np.unique(y_res, return_counts=True)))}")

X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)

# Reshape for CNN: (samples, timesteps, features)
X_train = X_train.values.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.values.reshape(X_test.shape[0],  X_test.shape[1],  1)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 2. Build CNN-LSTM Architecture

In [ ]:
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout

NUM_CLASSES  = len(label_encoder.classes_)
INPUT_SHAPE  = (X_train.shape[1], 1)

model = Sequential([
    Conv1D(64,  kernel_size=3, activation="relu", padding="same", input_shape=INPUT_SHAPE),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    Conv1D(128, kernel_size=3, activation="relu", padding="same"),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 3. Train

In [ ]:
import os
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

os.makedirs("results/metrics",  exist_ok=True)
os.makedirs("results/figures",  exist_ok=True)

callbacks = [
    ModelCheckpoint("results/CNN_LSTM_best.keras",
                    monitor="val_accuracy", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=15,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=5, min_lr=1e-6, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

## 4. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"],     label="Training Accuracy")
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
axes[0].set_title("C. Training and Validation Accuracy_T&FD",
                  fontsize=11, fontweight="bold")
axes[0].set_xlabel("Epochs"); axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"],     label="Training Loss")
axes[1].plot(history.history["val_loss"], label="Validation Loss")
axes[1].set_title("Training and Validation Loss", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Epochs"); axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("results/figures/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Best val accuracy: {max(history.history['val_accuracy']):.4f}")

## 5. Confusion Matrix & Classification Report

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

y_pred_proba  = model.predict(X_test)
y_pred_labels = np.argmax(y_pred_proba, axis=1)

print(classification_report(y_test, y_pred_labels,
      target_names=label_encoder.classes_))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_labels),
            annot=True, fmt="d", cmap="Blues",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title("C. Confusion Matrix_T&FD", fontsize=11, fontweight="bold")
plt.xlabel("Predicted Labels", fontsize=10)
plt.ylabel("True Labels", fontsize=10)
plt.tight_layout()
plt.savefig("results/figures/confusion_matrix_best.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
colors = ["blue", "orange", "green", "red"]

plt.figure(figsize=(6, 5))
for i, (cls, color) in enumerate(zip(label_encoder.classes_, colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
    auc = roc_auc_score(y_test_bin[:, i], y_pred_proba[:, i])
    plt.plot(fpr, tpr, color=color, label=f"Class {cls} (AUC={auc:.2f})")

plt.plot([0,1],[0,1],"k--")
plt.title("A. ROC Curves — All Condition Classes", fontsize=11, fontweight="bold")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.legend(); plt.tight_layout()
plt.savefig("results/figures/roc_curves_all_classes.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Save Model

In [ ]:
import json

model.save("results/CNN_LSTM_final.h5")
with open("results/label_classes.json", "w") as f:
    json.dump(list(label_encoder.classes_), f)

print("Model  -> results/CNN_LSTM_final.h5")
print("Labels -> results/label_classes.json")